# Step 2 — Data Cleaning & Preprocessing

**Goal:** turn the raw API responses (nested JSON, wrong dtypes, zeros that mean
"unknown") into a flat, typed, analysis-ready table saved to `data/processed/`.

We work through each transformation *manually* here so you see what it does —
then at the end we run the whole thing through `src/cleaning.py`'s `clean_movies()`,
which composes the exact same steps into one reusable, **tested** function
(see `tests/test_cleaning.py`).

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.append("..")
from src.tmdb_client import load_raw
from src.cleaning import (
    extract_names, extract_collection_name, extract_director,
    clean_movies, validate, DROP_COLUMNS,
)

raw = load_raw("../data/raw/movies_raw.json")   # never re-hits the API
df = pd.DataFrame(raw)
df.shape

(18, 28)

## 2.1 Drop irrelevant columns

`adult` and `video` are the same for every movie here (no information),
`imdb_id` duplicates `id`'s job, `original_title` mostly repeats `title`,
and `homepage` is unused in our analysis. Less noise from here on.

In [2]:
df = df.drop(columns=DROP_COLUMNS)
df.columns.tolist()

['backdrop_path',
 'belongs_to_collection',
 'budget',
 'genres',
 'id',
 'origin_country',
 'original_language',
 'overview',
 'popularity',
 'poster_path',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'softcore',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'vote_average',
 'vote_count',
 'credits']

## 2.2 Flatten the nested JSON columns

Look at what a single `genres` cell actually contains — a Python **list of dicts**,
not text:

In [3]:
df["genres"].iloc[0]

[{'id': 12, 'name': 'Adventure'},
 {'id': 878, 'name': 'Science Fiction'},
 {'id': 28, 'name': 'Action'}]

Vectorized column math can't help with arbitrary Python objects — this is the
legitimate use case for **`.apply()`**: run a function on every cell of a Series.
Our function is `extract_names` (from `src/cleaning.py`):

```python
extract_names([{'name': 'Action'}, {'name': 'Adventure'}])  ->  'Action|Adventure'
extract_names(None)                                          ->  NaN
```

In [4]:
df["belongs_to_collection"] = df["belongs_to_collection"].apply(extract_collection_name)

for col in ["genres", "production_countries", "production_companies"]:
    df[col] = df[col].apply(extract_names)

# spoken_languages: 'name' is the native spelling (e.g. 日本語); english_name reads better
df["spoken_languages"] = df["spoken_languages"].apply(lambda x: extract_names(x, key="english_name"))

df[["title", "genres", "belongs_to_collection", "spoken_languages"]].head()

,title,genres,belongs_to_collection,spoken_languages
0,Avengers: Endgame,Adventure|Science Fiction|Action,The Avengers Collection,English|Japanese|Xhosa
1,Avatar,Science Fiction|Action|Adventure,Avatar Collection,English|Spanish
2,Star Wars: The Force Awakens,Adventure|Action|Science Fiction,Star Wars Collection,English
3,Avengers: Infinity War,Adventure|Action|Science Fiction,The Avengers Collection,English|Xhosa
4,Titanic,Drama|Romance,NaN,English|French|German|Swedish|Italian|Russian


### Credits → cast, cast_size, director, crew_size

The `credits` field (which we got via `append_to_response`) holds two lists —
`cast` and `crew`. The final table needs four flat columns from it.

In [5]:
df["cast"] = df["credits"].apply(lambda c: extract_names(c.get("cast")) if isinstance(c, dict) else np.nan)
df["cast_size"] = df["credits"].apply(lambda c: len(c.get("cast", [])) if isinstance(c, dict) else 0)
df["director"] = df["credits"].apply(extract_director)
df["crew_size"] = df["credits"].apply(lambda c: len(c.get("crew", [])) if isinstance(c, dict) else 0)
df = df.drop(columns=["credits"])

df[["title", "director", "cast_size", "crew_size"]]

,title,director,cast_size,crew_size
0,Avengers: Endgame,Joe Russo,105,613
1,Avatar,James Cameron,67,999
2,Star Wars: The Force Awakens,J.J. Abrams,99,270
3,Avengers: Infinity War,Anthony Russo,69,736
4,Titanic,James Cameron,117,270
5,Jurassic World,Colin Trevorrow,53,420
6,The Lion King,Jon Favreau,20,60
7,The Avengers,Joss Whedon,112,640
8,Furious 7,James Wan,50,239
9,Avengers: Age of Ultron,Joss Whedon,75,660


## 2.3 Inspect with `value_counts()`

Frequency counts are the fastest anomaly detector: typos, placeholders and
suspicious duplicates jump out immediately.

In [6]:
print(df["belongs_to_collection"].value_counts(dropna=False), "\n")
print(df["original_language"].value_counts(), "\n")
print(df["status"].value_counts())

belongs_to_collection
The Avengers Collection                4
Star Wars Collection                   2
NaN                                    2
Jurassic Park Collection               2
Frozen Collection                      2
Avatar Collection                      1
The Lion King (Reboot) Collection      1
The Fast and the Furious Collection    1
Black Panther Collection               1
Harry Potter Collection                1
The Incredibles Collection             1
Name: count, dtype: int64 

original_language
en    18
Name: count, dtype: int64 

status
Released    18
Name: count, dtype: int64


## 2.4 Fix datatypes

`errors="coerce"` is the key idea: anything unparseable becomes `NaN` instead of
crashing the pipeline. A datetime dtype unlocks things like `.dt.year` later.

In [7]:
for col in ["budget", "id", "popularity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

df.dtypes

backdrop_path                    object
belongs_to_collection            object
budget                            int64
genres                           object
id                                int64
origin_country                   object
original_language                object
overview                         object
popularity                      float64
poster_path                      object
production_companies             object
production_countries             object
release_date             datetime64[ns]
revenue                           int64
runtime                           int64
softcore                           bool
spoken_languages                 object
status                           object
tagline                          object
title                            object
vote_average                    float64
vote_count                        int64
cast                             object
cast_size                         int64
director                         object


## 2.5 Zeros are lies, and dollars are too big

A movie with budget `0` wasn't free to make — the data is just **missing**, and
leaving it as 0 would poison every average and ROI we compute later. `NaN` is
honest: Pandas automatically excludes it from means, sums and plots.

We also rescale budget/revenue to **millions of USD** — easier to read, and it's
what the final column names (`budget_musd`) promise.

In [8]:
df[["budget", "revenue", "runtime"]] = df[["budget", "revenue", "runtime"]].replace(0, np.nan)

df["budget_musd"] = df["budget"] / 1e6
df["revenue_musd"] = df["revenue"] / 1e6
df = df.drop(columns=["budget", "revenue"])

# A vote_average computed from 0 votes is noise, not signal
df.loc[df["vote_count"] == 0, "vote_average"] = np.nan

# Placeholder text -> proper missing values
df[["overview", "tagline"]] = df[["overview", "tagline"]].replace(["No Data", "No overview found.", ""], np.nan)

df[["budget_musd", "revenue_musd", "runtime", "vote_average"]].describe().round(1)

,budget_musd,revenue_musd,runtime,vote_average
count,18.0,18.0,18.0,18.0
mean,213.8,1691.9,138.0,7.4
std,62.0,521.0,23.8,0.5
min,125.0,1243.2,102.0,6.5
25%,162.5,1336.2,125.2,7.1
50%,200.0,1484.5,135.5,7.3
75%,243.0,1957.2,147.5,7.8
max,356.0,2923.7,194.0,8.2


## 2.6 Row-level filters

- drop duplicate ids and rows missing `id`/`title`
- keep only rows with **≥ 10 non-NaN values** (`dropna(thresh=10)`) — a row
  that's mostly holes can't support analysis
- keep only `Released` movies, then `status` carries no information → drop it

In [9]:
before = len(df)
df = df.drop_duplicates(subset="id").dropna(subset=["id", "title"])
df = df.dropna(thresh=10)
df = df[df["status"] == "Released"].drop(columns=["status"])
print(f"{before} rows -> {len(df)} rows")

18 rows -> 18 rows


## 2.7 Final shape: reorder columns, reset index

In [10]:
column_order = [
    "id", "title", "tagline", "release_date", "genres", "belongs_to_collection",
    "original_language", "budget_musd", "revenue_musd", "production_companies",
    "production_countries", "vote_count", "vote_average", "popularity", "runtime",
    "overview", "spoken_languages", "poster_path", "cast", "cast_size",
    "director", "crew_size",
]
df = df[column_order].reset_index(drop=True)
df.head(3)

,id,title,tagline,release_date,genres,belongs_to_collection,original_language,budget_musd,revenue_musd,production_companies,...,vote_average,popularity,runtime,overview,spoken_languages,poster_path,cast,cast_size,director,crew_size
0,299534,Avengers: Endgame,Avenge the fallen.,2019-04-24,Adventure|Science Fiction|Action,The Avengers Collection,en,356.0,2799.439100,Marvel Studios,...,8.239,22.5075,181,After the devastating events of Avengers: Infi...,English|Japanese|Xhosa,/ulzhLuWrPK07P1YkdWQLZnQh1JL.jpg,Robert Downey Jr.|Chris Evans|Mark Ruffalo|Chr...,105,Joe Russo,613
1,19995,Avatar,Enter the world of Pandora.,2009-12-16,Science Fiction|Action|Adventure,Avatar Collection,en,237.0,2923.706026,Dune Entertainment|Lightstorm Entertainment|20...,...,7.609,23.9453,162,"In the 22nd century, a paraplegic Marine is di...",English|Spanish,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,Sam Worthington|Zoe Saldaña|Sigourney Weaver|S...,67,James Cameron,999
2,140607,Star Wars: The Force Awakens,Every generation has a story.,2015-12-15,Adventure|Action|Science Fiction,Star Wars Collection,en,245.0,2068.223624,Lucasfilm Ltd.|Bad Robot,...,7.248,10.1950,136,Thirty years after defeating the Galactic Empi...,English,/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg,Daisy Ridley|John Boyega|Oscar Isaac|Adam Driv...,99,J.J. Abrams,270


## 2.8 The whole pipeline, reproducibly

Everything above lives as one tested function. Running it on the untouched raw
data must give the same result as our manual walk-through — proof the pipeline
is deterministic and the module matches the notebook.

In [11]:
movies = clean_movies(load_raw("../data/raw/movies_raw.json"))

print("identical to manual result:", movies.equals(df))
validate(movies)   # data checks: unique ids, positive budgets, datetime dtype, ...

identical to manual result: True
All validation checks passed ✔  (18 movies, 22 columns)


In [12]:
movies.to_csv("../data/processed/movies_clean.csv", index=False)
print("saved -> data/processed/movies_clean.csv")

saved -> data/processed/movies_clean.csv


**Note for the next notebooks:** CSV stores everything as text, so when loading
we'll pass `parse_dates=["release_date"]` to get the datetime dtype back.

➡️ **Next: `03_kpi_analysis.ipynb`** — profit, ROI, the ranking UDF, and groupby.